# Pandas Indicator Tester (One-by-One)

Pure **pandas/numpy** technical indicators — no TA-Lib required.

Run setup cells once, then run any indicator cell below.

```bash
pip install -e ".[notebook]"
jupyter lab notebooks/pandas_indicator_tester.ipynb
```


In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from indicator_testing.data_loader import describe_ohlc, load_ohlc
from indicator_testing.pandas_indicators import PandasIndicatorRegistry
from indicator_testing.pandas_runner import run_pandas_indicator

print(f"Project root: {PROJECT_ROOT}")


## Configuration


In [ ]:
CSV_PATH = PROJECT_ROOT / "questdb-query-1781940224994.csv"
RESAMPLE = "none"   # "none" | "daily" | "monthly" | "weekly"
SYMBOL = None


In [ ]:
resample_arg = None if RESAMPLE == "none" else RESAMPLE
df = load_ohlc(CSV_PATH, resample=resample_arg, symbol=SYMBOL, warn_short=False)
registry = PandasIndicatorRegistry()
info = describe_ohlc(df)

print(f"Loaded {info['bars']} bars | {info['inferred_frequency']}")
print(f"Pandas indicators: {len(registry.all_names())}")
df.tail(3)


## Helpers


In [ ]:
def show_validation(result):
    meta = registry.get(result.name)
    print(f"{'='*60}")
    print(f"{result.name}  |  {result.group}  (pandas)")
    print(f"Status: {result.status}  |  Warmup: {result.warmup_bars}")
    print(f"Outputs: {meta.output_names}  |  Params: {result.params_used}")
    if result.message and result.status != "success":
        print(f"Message: {result.message}")
    if result.validation:
        tag = "PASS" if result.validation.passed else "FAIL"
        print(f"Validation: {tag}")
        for c in result.validation.checks:
            mark = "OK" if c.passed else "X"
            print(f"  [{mark}] {c.name}: {c.message}")
    print(f"{'='*60}")


def result_to_frame(result, n_tail=15):
    if result.status != "success" or not result.outputs:
        return pd.DataFrame()
    out = pd.DataFrame({"close": df["close"]})
    for k, arr in result.outputs.items():
        out[k] = arr
    out.index = df.index
    return out.tail(n_tail)


def plot_pandas(result):
    if result.status != "success" or not result.outputs:
        print("Nothing to plot.")
        return
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(df.index, df["close"], label="Close", color="black", lw=1)
    axes[0].set_ylabel("Price")
    axes[0].legend(loc="upper left")
    for key, arr in result.outputs.items():
        axes[1].plot(df.index, arr, label=key, lw=1)
    axes[1].set_ylabel(result.name)
    axes[1].legend(loc="upper left")
    fig.suptitle(f"{result.name} (pandas) — {result.params_used}")
    fig.tight_layout()
    plt.show()


def test_pandas_indicator(name, params=None):
    result = run_pandas_indicator(name, df, registry, params=params)
    show_validation(result)
    display(result_to_frame(result))
    plot_pandas(result)
    return result


## Table of contents

- [Overlap Studies](#overlap-studies)
- [Momentum Indicators](#momentum-indicators)
- [Volatility Indicators](#volatility-indicators)
- [Volume Indicators](#volume-indicators)
- [Price Transform](#price-transform)
- [Statistic Functions](#statistic-functions)


<a id="overlap-studies"></a>
## Overlap Studies (7)


### SMA
Outputs: `['sma']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("SMA")


### EMA
Outputs: `['ema']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("EMA")


### WMA
Outputs: `['wma']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("WMA")


### DEMA
Outputs: `['dema']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("DEMA")


### TEMA
Outputs: `['tema']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("TEMA")


### HMA
Outputs: `['hma']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("HMA")


### VWMA
Outputs: `['vwma']` · Defaults: `{'length': 20}` · requires volume


In [ ]:
test_pandas_indicator("VWMA")


<a id="momentum-indicators"></a>
## Momentum Indicators (12)


### RSI
Outputs: `['rsi']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("RSI")


### MACD
Outputs: `['macd', 'macdsignal', 'macdhist']` · Defaults: `{'fast': 12, 'slow': 26, 'signal': 9}`


In [ ]:
test_pandas_indicator("MACD")


### ROC
Outputs: `['roc']` · Defaults: `{'length': 10}`


In [ ]:
test_pandas_indicator("ROC")


### MOM
Outputs: `['mom']` · Defaults: `{'length': 10}`


In [ ]:
test_pandas_indicator("MOM")


### STOCH
Outputs: `['slowk', 'slowd']` · Defaults: `{'k': 14, 'd': 3, 'smooth_k': 3}`


In [ ]:
test_pandas_indicator("STOCH")


### STOCHRSI
Outputs: `['stochrsi_k', 'stochrsi_d']` · Defaults: `{'length': 14, 'k': 3, 'd': 3}`


In [ ]:
test_pandas_indicator("STOCHRSI")


### CCI
Outputs: `['cci']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("CCI")


### WILLR
Outputs: `['willr']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("WILLR")


### MFI
Outputs: `['mfi']` · Defaults: `{'length': 14}` · requires volume


In [ ]:
test_pandas_indicator("MFI")


### CMO
Outputs: `['cmo']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("CMO")


### PPO
Outputs: `['ppo']` · Defaults: `{'fast': 12, 'slow': 26}`


In [ ]:
test_pandas_indicator("PPO")


### TRIX
Outputs: `['trix']` · Defaults: `{'length': 15}`


In [ ]:
test_pandas_indicator("TRIX")


<a id="volatility-indicators"></a>
## Volatility Indicators (5)


### BBANDS
Outputs: `['upperband', 'middleband', 'lowerband']` · Defaults: `{'length': 20, 'std': 2.0}`


In [ ]:
test_pandas_indicator("BBANDS")


### ATR
Outputs: `['atr']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("ATR")


### NATR
Outputs: `['natr']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("NATR")


### TRANGE
Outputs: `['trange']` · Defaults: `{}`


In [ ]:
test_pandas_indicator("TRANGE")


### DONCHIAN
Outputs: `['donchian_upper', 'donchian_mid', 'donchian_lower']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("DONCHIAN")


<a id="volume-indicators"></a>
## Volume Indicators (5)


### OBV
Outputs: `['obv']` · Defaults: `{}` · requires volume


In [ ]:
test_pandas_indicator("OBV")


### AD
Outputs: `['ad']` · Defaults: `{}` · requires volume


In [ ]:
test_pandas_indicator("AD")


### ADOSC
Outputs: `['adosc']` · Defaults: `{'fast': 3, 'slow': 10}` · requires volume


In [ ]:
test_pandas_indicator("ADOSC")


### CMF
Outputs: `['cmf']` · Defaults: `{'length': 20}` · requires volume


In [ ]:
test_pandas_indicator("CMF")


### VWAP
Outputs: `['vwap']` · Defaults: `{}` · requires volume


In [ ]:
test_pandas_indicator("VWAP")


<a id="price-transform"></a>
## Price Transform (4)


### AVGPRICE
Outputs: `['avgprice']` · Defaults: `{}`


In [ ]:
test_pandas_indicator("AVGPRICE")


### MEDPRICE
Outputs: `['medprice']` · Defaults: `{}`


In [ ]:
test_pandas_indicator("MEDPRICE")


### TYPPRICE
Outputs: `['typprice']` · Defaults: `{}`


In [ ]:
test_pandas_indicator("TYPPRICE")


### WCLPRICE
Outputs: `['wclprice']` · Defaults: `{}`


In [ ]:
test_pandas_indicator("WCLPRICE")


<a id="statistic-functions"></a>
## Statistic Functions (4)


### STDDEV
Outputs: `['stddev']` · Defaults: `{'length': 5}`


In [ ]:
test_pandas_indicator("STDDEV")


### VAR
Outputs: `['var']` · Defaults: `{'length': 5}`


In [ ]:
test_pandas_indicator("VAR")


### ZSCORE
Outputs: `['zscore']` · Defaults: `{'length': 20}`


In [ ]:
test_pandas_indicator("ZSCORE")


### LINEARREG
Outputs: `['linearreg']` · Defaults: `{'length': 14}`


In [ ]:
test_pandas_indicator("LINEARREG")
